<a href="https://colab.research.google.com/github/PeiHsiuLu/112-2-Programming-Language/blob/main/gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow gradio opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 70.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.3/316.3 kB 30.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 80.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 15.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 6.1 MB/s e

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import gradio as gr

# Load the pre-trained model
model_path = '/content/model.h5'
model = load_model(model_path)

# Emotions dictionary
emotions = {
    0: "Angry", 1: "Fear", 2: "Happy",
    3: "Neutral", 4: "Sad", 5: "Surprise"
}

# Define the function to process an image and predict emotion
def predict_emotion(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = faceCascade.detectMultiScale(gray, 1.1, 4)
    emotion_detected = "None"  # Default if no face is detected

    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        face_roi = cv2.resize(face_roi, (224, 224))
        face_roi = cv2.cvtColor(face_roi, cv2.COLOR_GRAY2RGB)  # Convert grayscale to RGB
        face_roi = np.expand_dims(face_roi, axis=0) / 255.0

        try:
            predictions = model.predict(face_roi)
            emotion_index = np.argmax(predictions)
            if emotion_index in emotions:
                emotion_detected = emotions[emotion_index]
        except Exception as e:
            print(f"Error during model prediction: {e}")

    return emotion_detected

# Define the function to process video and predict emotion for each frame
def predict_emotion_from_video(video_path):
    cap = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    output_path = '/mnt/data/output.avi'
    out = cv2.VideoWriter(output_path, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_with_emotion = predict_emotion(frame)
        out.write(frame_with_emotion)

    cap.release()
    out.release()

    return output_path

# Set up the Gradio interfaces
image_interface = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=gr.Textbox(label="Detected Emotion from Image"),
    title="Emotion Detection from Image",
    description="Upload an image and the model will predict the emotion."
)

video_interface = gr.Interface(
    fn=predict_emotion_from_video,
    inputs=gr.Video(label="Upload Video"),
    outputs=gr.Video(label="Detected Emotion from Video"),
    title="Emotion Detection from Video",
    description="Upload a video and the model will predict the emotion for each frame."
)

# Combine the interfaces
demo = gr.TabbedInterface(
    [image_interface, video_interface],
    ["Image Upload", "Video Upload"]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://114580c129050752d4.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
